# Dusk vs V1 historical counterfactual benchmark

This notebook is a bounded, reader-facing view of one verified `ResultManifestV1`. Its executor validates and injects the manifest before any code cell can run.

## tl;dr

The lines below identify the sealed run, evidence scope, certification state, shared-profile decision, holdout interval, and mandatory coverage gates. They are generated from the validated manifest and make no forecast claim.

In [ ]:
result_scope = metrics_manifest["scope"]
scope_label = "selected-panel"
if result_scope["scope_mode"] == "full_universe":
    scope_label = "full-universe"
winner_label = f"{scope_label} shared-profile winner={metrics_manifest['global_winner_profile_id']}"
if metrics_manifest["no_global_winner"]:
    winner_label = f"no {scope_label} shared-profile winner"
print(f"Run: {metrics_manifest['run_id']}")
print(f"Dataset: {metrics_manifest['dataset_id']}")
print(f"Evidence scope: {scope_label}; tier={result_scope['evidence_tier']}; certification={result_scope['final_phase_gate_decision']['usd_certification']}")
print(f"Decision state: {winner_label}")
print(f"Holdout [UTC unix seconds): {metrics_manifest['holdout_start_unix_seconds']} .. {metrics_manifest['holdout_end_unix_seconds_exclusive']}")
for coverage_name in ("visible_swap_volume", "borrowed_notional", "revenue_notional"):
    coverage_value = metrics_manifest["coverage"][coverage_name]
    print(f"Coverage {coverage_name}: {coverage_value['covered_numerator']}/{coverage_value['total_denominator']}; {coverage_value['actual_bps']} bps; passes={coverage_value['passes']}")

## Context & Methods

V1 is the factual historical control and Dusk is the counterfactual replay. This notebook does not read raw events, prices, configuration, or network resources: the sealed metrics manifest is its only benchmark data input. All available metrics remain exact signed numerator / positive denominator strings.

### Key Assumptions

- Results describe the manifest's fixed historical holdout, not a forecast.
- A shared-profile winner is reported only within the manifest's explicit scope and mutually exclusive winner/no-winner state.
- DecisionGrade selected-panel evidence may retain only typed decoder-revision and terminal-state limitations; all other source, replay, metric, and conservation gates remain mandatory.
- Unavailable metrics are labels, never zeros.
- Synthetic arbitrage and synthetic protocol settlement are separately labeled and are not user-flow revenue.
- Preview limits affect display only; the manifest checksum commits to the full evidence.

In [ ]:
print(f"Metrics schema: {metrics_manifest['schema_version']}")
print(f"Manifest self-hash: {metrics_manifest['manifest_sha256']}")
print(f"Holdout evidence hash: {metrics_manifest['holdout_evidence_sha256']}")
print(f"Universe window hash: {metrics_manifest['universe_window_sha256']}")
print(f"Dusk source hash: {metrics_manifest['dusk_source_sha256']}")
print(f"Scope mode: {result_scope['scope_mode']}")
print(f"Selected USD-headline markets: {len(result_scope['selected_markets'])}")
print(f"Selected-market set hash: {result_scope['selected_market_set_sha256']}")
print(f"Final phase-gate hash: {result_scope['final_phase_gate_sha256']}")
representativeness = result_scope['cohort_representativeness']
if representativeness is not None:
    for activity_name in ("swaps", "debt_actions", "liquidations"):
        activity = representativeness[activity_name]
        print(f"Selected-panel {activity_name}: {activity['selected_panel_rows']}/{activity['visible_universe_rows']} rows; {activity['selected_panel_coverage_bps']} bps")
print(f"Headline hLP scenario: {metrics_manifest['headline_hlp_scenario']}")
print(f"No-hLP control included: {metrics_manifest['no_hlp_control_included']}")
print(f"Per-market calibration qualification hash: {metrics_manifest['per_market_calibration_qualification_sha256']}")
print(f"Per-market holdout qualification hash: {metrics_manifest['per_market_holdout_qualification_sha256']}")
print(f"Exact transaction fetch: {metrics_manifest['coverage']['exact_transaction_fetch_bps']} bps")
print(f"Exact decode: {metrics_manifest['coverage']['exact_decode_bps']} bps")
print(f"V1 zero-difference reconciled: {metrics_manifest['coverage']['v1_zero_difference_reconciled']}")

## Data

This section profiles the injected manifest at its declared grain: aggregate metrics plus per-market raw and historical-USD metrics. Previews are deterministic and bounded.

In [ ]:
PREVIEW_LIMIT = 12

def exact_text(value):
    if value["availability"] != "available":
        return f"unavailable:{value['availability']} {value['unit']}"
    return f"{value['numerator']}/{value['denominator']} {value['unit']}"

def collected_metrics():
    rows = []
    for metric in metrics_manifest["aggregate_metrics"]:
        rows.append(("aggregate", metric))
    for market in metrics_manifest["markets"]:
        for metric in market["raw_metrics"]:
            rows.append((f"market:{market['market']}:raw", metric))
        for metric in market["usd_metrics"]:
            rows.append((f"market:{market['market']}:usd", metric))
    return rows

metric_rows = collected_metrics()
availability_counts = {}
for scope, metric in metric_rows:
    for side_name in ("v1", "dusk", "paired_delta"):
        state = metric[side_name]["availability"]
        availability_counts[state] = availability_counts.get(state, 0) + 1
print(f"Markets: {len(metrics_manifest['markets'])}")
print(f"Aggregate metrics: {len(metrics_manifest['aggregate_metrics'])}")
print(f"All named metrics: {len(metric_rows)}")
print(f"Bootstrap intervals: {len(metrics_manifest['bootstrap_intervals'])}")
for state in sorted(availability_counts):
    print(f"Metric-value availability {state}: {availability_counts[state]}")

In [ ]:
source_names = sorted(metrics_manifest["source_artifact_sha256"])
print(f"Source artifact hashes ({len(source_names)} total; showing at most {PREVIEW_LIMIT}):")
for source_name in source_names[:PREVIEW_LIMIT]:
    print(f"  {source_name}: {metrics_manifest['source_artifact_sha256'][source_name]}")
if len(source_names) > PREVIEW_LIMIT:
    print(f"  ... {len(source_names) - PREVIEW_LIMIT} more committed source artifacts")

## Results

The exact values are grouped around the benchmark's requested decisions: collateral efficiency, spot capital efficiency, leverage, LP/protocol revenue, safety, and coverage. A positive paired delta always means `Dusk - V1`; whether positive is desirable depends on the metric definition.

In [ ]:
family_order = ("collateral_efficiency", "spot_capital_efficiency", "leverage", "lp_revenue", "protocol_revenue", "balance_sheet", "safety", "coverage")
for family_name in family_order:
    family_metrics = sorted((metric for metric in metrics_manifest["aggregate_metrics"] if metric["family"] == family_name), key=lambda metric: metric["metric_id"])
    print(f"[{family_name}] {len(family_metrics)} aggregate metrics")
    for metric in family_metrics[:6]:
        print(f"  {metric['metric_id']}")
        print(f"    V1={exact_text(metric['v1'])}")
        print(f"    Dusk={exact_text(metric['dusk'])}")
        print(f"    Delta={exact_text(metric['paired_delta'])}")
    if len(family_metrics) > 6:
        print(f"  ... {len(family_metrics) - 6} more {family_name} aggregate metrics")

In [ ]:
market_rows = sorted(((scope, metric) for scope, metric in metric_rows if scope != "aggregate"), key=lambda row: (row[0], row[1]["metric_id"]))
print(f"Per-market metric preview ({len(market_rows)} total; showing at most {PREVIEW_LIMIT}):")
for scope, metric in market_rows[:PREVIEW_LIMIT]:
    print(f"  {scope} | {metric['family']} | {metric['metric_id']} | delta={exact_text(metric['paired_delta'])}")
if len(market_rows) > PREVIEW_LIMIT:
    print(f"  ... {len(market_rows) - PREVIEW_LIMIT} more per-market metrics")

In [ ]:
intervals = sorted(metrics_manifest["bootstrap_intervals"], key=lambda interval: interval["metric_id"])
print(f"Paired bootstrap intervals ({len(intervals)} total; showing at most {PREVIEW_LIMIT}):")
for interval in intervals[:PREVIEW_LIMIT]:
    print(f"  {interval['metric_id']}: [{interval['lower_025']}, {interval['median']}, {interval['upper_975']}] {interval['unit']}; block_days={interval['block_days']}; samples={interval['samples']}")
if len(intervals) > PREVIEW_LIMIT:
    print(f"  ... {len(intervals) - PREVIEW_LIMIT} more bootstrap intervals")
checks = metrics_manifest["accounting_verification"]
print(f"Accounting checks: V1={checks['v1_conservation_checks']}, Dusk transitions={checks['dusk_atomic_transition_checks']}, paired revenue lanes={checks['paired_revenue_lane_checks']}; failures={checks['v1_conservation_failures'] + checks['dusk_atomic_transition_failures'] + checks['paired_revenue_lane_failures']}")

## Takeaways

This final cell gives only mechanically supported directional counts, the sealed decision state, and bounded limitations. Read each delta using its definition; no generic sign convention can turn all metric families into wins or losses.

In [ ]:
available_deltas = [metric for metric in metrics_manifest["aggregate_metrics"] if metric["paired_delta"]["availability"] == "available"]
positive_count = sum(1 for metric in available_deltas if int(metric["paired_delta"]["numerator"]) > 0)
zero_count = sum(1 for metric in available_deltas if int(metric["paired_delta"]["numerator"]) == 0)
negative_count = sum(1 for metric in available_deltas if int(metric["paired_delta"]["numerator"]) < 0)
print(f"Available aggregate paired deltas: positive={positive_count}, zero={zero_count}, negative={negative_count}")
if metrics_manifest["no_global_winner"]:
    print(f"Decision: no {scope_label} shared-profile winner passed the sealed joint selection state.")
else:
    print(f"Decision: sealed {scope_label} shared-profile winner={metrics_manifest['global_winner_profile_id']}")
print(f"Pareto profiles: {', '.join(metrics_manifest['pareto_profile_ids']) if metrics_manifest['pareto_profile_ids'] else 'none'}")
gate_limitations = result_scope['final_phase_gate_decision']['limitations']
print(f"Certification limitations ({len(gate_limitations)}):")
for gate_limitation in gate_limitations:
    print(f"  - {gate_limitation['gate']}: {gate_limitation['detail_code']}")
limitations = metrics_manifest["limitations"]
print(f"Limitations ({len(limitations)} total; showing at most 5):")
for limitation in limitations[:5]:
    print(f"  - {limitation}")
if len(limitations) > 5:
    print(f"  ... {len(limitations) - 5} more limitations in the sealed manifest")